In [1]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()
pp.load_config('default_config.toml')

# Enable logging when initializing
pp.init(log_level="DEBUG")  # or "INFO", "WARNING", "ERROR"

# Or configure separately
pp.configure_logging(level="INFO")

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGA</cre>ATTACGG<bc/>AGATCGGA')\
                  .named('template_pool')

mutated_pool = template_pool.stylize(region='cre', style='goldenrod')\
                            .mutagenize(region='cre',
                                        num_mutations=2, 
                                        style='yellow bold swapcase',
                                        mode='random', 
                                        num_states=5,
                                        prefix='mutagenize').named('mutated_pool')\
                            .repeat(2, prefix='v', iter_order=-2)

L = len('GGAAAGCGGGCAGTGAGCACACAGGA') 
recomb_pool = template_pool.recombine(region='cre', 
                                      num_breakpoints=3,
                                      sources=['A'*L, 'C'*L, 'G'*L, 'T'*L],
                                      styles=['palegreen', 'springgreen', 'limegreen', 'forestgreen'],
                                      style_by='order',
                                      mode='random',
                                      num_states=5,
                                      prefix='recomb').named('recomb_pool')\
                            .repeat(2, prefix='v', iter_order=-2)


deletion_pool = template_pool.stylize(region='cre', style='salmon')\
                             .deletion_scan(region='cre', 
                                            deletion_length=6, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            style='red bold',
                                            prefix='delscan').named('deletion_pool')\
                            .repeat(2, prefix='v', iter_order=-2)

sites_pool=pp.from_seqs(['AAAAAA','TTTTTT'], 
                        mode='sequential', 
                        iter_order=-1).named('sites_pool')

insertion_pool = template_pool.stylize(region='cre', style='blue')\
                              .insertion_scan(region='cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential',
                                              prefix='insscan',
                                              prefix_position='pos', 
                                              prefix_insert='ins',
                                              style='cyan bold swapcase').named('insertion_pool')
                              
shuffle_pool = template_pool.stylize(region='cre', style='purple')\
                            .shuffle_scan(region='cre', 
                                          shuffle_length=6, 
                                          shuffles_per_position=2,
                                          positions=slice(None, None, 5), 
                                          mode='sequential', 
                                          style='magenta bold swapcase',
                                          prefix='shufscan',
                                          prefix_position='pos',
                                          prefix_shuffle='shuf')
                            

combo_pool = pp.stack([mutated_pool, recomb_pool, deletion_pool, insertion_pool, shuffle_pool])\
    .named('stack_pool')\
    .insert_kmers(region='bc', mode='random', length=5, prefix='bc', style='green bold')\
    .named('combo_pool')\
    .stylize(which='tags', style='gray')

pp.toggle_styles(on=True)
combo_pool.print_library(show_name=True, seed=42)

INFO - poolparty.party - Initialized default Party
INFO - poolparty.generate_library - Starting library generation: pool=pool[26] num_seqs=50 seed=42
INFO - poolparty.generate_library - Completed library generation: 50 sequences


pool[26]: seq_length=None, num_states=50
state  name                           seq
    0  mutagenize_0.v_0.bc_00         TCCCGACT<cre>GGAAAaCaGGCAGTGAGCACACAGGA</cre>ATTACGG<bc>CTCTG</bc>AGATCGGA
    1  mutagenize_0.v_1.bc_01         TCCCGACT<cre>GGAAAaCaGGCAGTGAGCACACAGGA</cre>ATTACGG<bc>AATGT</bc>AGATCGGA
    2  mutagenize_1.v_0.bc_02         TCCCGACT<cre>tGAAAGCGGGCAtTGAGCACACAGGA</cre>ATTACGG<bc>TCTGG</bc>AGATCGGA
    3  mutagenize_1.v_1.bc_03         TCCCGACT<cre>tGAAAGCGGGCAtTGAGCACACAGGA</cre>ATTACGG<bc>GGCTG</bc>AGATCGGA
    4  mutagenize_2.v_0.bc_04         TCCCGACT<cre>GGAAAGCGGcgAGTGAGCACACAGGA</cre>ATTACGG<bc>TCAAA</bc>AGATCGGA
    5  mutagenize_2.v_1.bc_05         TCCCGACT<cre>GGAAAGCGGcgAGTGAGCACACAGGA</cre>ATTACGG<bc>TCGTG</bc>AGATCGGA
    6  mutagenize_3.v_0.bc_06         TCCCGACT<cre>cGAAAGgGGGCAGTGAGCACACAGGA</cre>ATTACGG<bc>CCTTT</bc>AGATCGGA
    7  mutagenize_3.v_1.bc_07         TCCCGACT<cre>cGAAAGgGGGCAGTGAGCACACAGGA</cre>ATTACGG<bc>GTTAA</bc>AGATCGGA
    8  mutage

DnaPool(id=26, name='pool[26]', op='op[26]:stylize', num_states=50)

In [2]:
pp.toggle_cards(on=False)
df = combo_pool.generate_library(report_design_cards=True)
print(df.columns)
df.tail()

INFO - poolparty.generate_library - Starting library generation: pool=pool[26] num_seqs=50 seed=None
INFO - poolparty.generate_library - Completed library generation: 50 sequences


Index(['name', 'seq'], dtype='object')


,name,seq
45,shufscan_5.pos_2.shuf_1.bc_95,TCCCGACT<cre>GGAAAGCGGGGATGCAGCACACAGGA</cre>A...
46,shufscan_6.pos_3.shuf_0.bc_96,TCCCGACT<cre>GGAAAGCGGGCAGTGCAAACGCAGGA</cre>A...
47,shufscan_7.pos_3.shuf_1.bc_97,TCCCGACT<cre>GGAAAGCGGGCAGTGCGACAACAGGA</cre>A...
48,shufscan_8.pos_4.shuf_0.bc_98,TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACAAGAGC</cre>A...
49,shufscan_9.pos_4.shuf_1.bc_99,TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACGCGAAA</cre>A...


In [3]:
combo_pool.print_dag()

pool[26] (pool, n=50)
└── op[26]:stylize [mode=fixed, n=1]
    └── combo_pool (pool, n=50)
        └── op[25]:get_kmers [mode=random, n=1]
            └── stack_pool (pool, n=50)
                └── op[24]:stack [mode=sequential, n=5]
                    ├── pool[3] (pool, n=10)
                    │   └── op[3]:repeat [mode=sequential, n=2]
                    │       └── mutated_pool (pool, n=5)
                    │           └── op[2]:mutagenize [mode=random, n=5]
                    │               └── pool[1] (pool, n=1)
                    │                   └── op[1]:stylize [mode=fixed, n=1]
                    │                       └── template_pool (pool, n=1)
                    │                           └── op[0]:from_seq [mode=fixed, n=1]
                    ├── pool[9] (pool, n=10)
                    │   └── op[9]:repeat [mode=sequential, n=2]
                    │       └── recomb_pool (pool, n=5)
                    │           └── op[8]:recombine [mode=random, n

DnaPool(id=26, name='pool[26]', op='op[26]:stylize', num_states=50)